## Inference + evaluation (Colab)

Compares base vs adapter (LoRA/QLoRA) on a batch of missions.

Assumes:
- `00_colab_setup.ipynb` was executed
- (optional) `03_sft_peft_train_colab.ipynb` produced an adapter under `artifacts/`.


In [ ]:
from pathlib import Path
import sys

BENCHMARK_ROOT = Path('/content/nav4rails/repositories/FineTuningOnTelecomCluster/benchmark').resolve()
if str(BENCHMARK_ROOT) not in sys.path:
    sys.path.insert(0, str(BENCHMARK_ROOT))

SCRIPTS_DIR = BENCHMARK_ROOT / 'scripts'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print('Benchmark root:', BENCHMARK_ROOT)


In [ ]:
# Evaluate base model vs adapter on synthetic missions

import json
from src.config_loader import load_experiment_config
from src.evaluation.metrics import write_reports

# Build a mission list
from src.data.synthetic_generator import PROMPT_TEMPLATES
missions = PROMPT_TEMPLATES * 10

# Config for evaluation (prompting)
cfg = load_experiment_config(BENCHMARK_ROOT / 'configs' / 'prompt_zero_shot.yaml')
cfg.output_root = str(BENCHMARK_ROOT / 'runs' / 'colab_eval_base')

# Optionally set adapter path for adapted run (must point to a saved adapter dir)
adapter_path = None  # e.g. str(BENCHMARK_ROOT / 'artifacts' / 'sft_lora')

from src.evaluation.runner import ExperimentRunner
from src.models.factory import load_model_bundle
from _hf_generate import HFChatGenerator

# Base run
runner = ExperimentRunner(cfg)
model, tok = load_model_bundle(cfg.model, cfg.peft)
gen = HFChatGenerator(model=model, tokenizer=tok, device=str(getattr(model,'device',None)) or None)

rows = []
for m in missions[:20]:
    out = runner.run_prompt_experiment(
        mission=m,
        generate_fn=lambda msgs: gen.generate(msgs, max_new_tokens=cfg.generation.max_new_tokens, temperature=cfg.generation.temperature, top_p=cfg.generation.top_p, do_sample=cfg.generation.do_sample),
    )
    rows.append(out['metrics'])

print('base rows:', len(rows))
print(write_reports(cfg.output_root, rows))

# Adapter run (if provided)
if adapter_path:
    cfg2 = load_experiment_config(BENCHMARK_ROOT / 'configs' / 'prompt_zero_shot.yaml')
    cfg2.output_root = str(BENCHMARK_ROOT / 'runs' / 'colab_eval_adapter')
    cfg2.peft.adapter_path = adapter_path
    runner2 = ExperimentRunner(cfg2)
    model2, tok2 = load_model_bundle(cfg2.model, cfg2.peft)
    gen2 = HFChatGenerator(model=model2, tokenizer=tok2, device=str(getattr(model2,'device',None)) or None)
    rows2 = []
    for m in missions[:20]:
        out = runner2.run_prompt_experiment(
            mission=m,
            generate_fn=lambda msgs: gen2.generate(msgs, max_new_tokens=cfg2.generation.max_new_tokens, temperature=cfg2.generation.temperature, top_p=cfg2.generation.top_p, do_sample=cfg2.generation.do_sample),
        )
        rows2.append(out['metrics'])
    print('adapter rows:', len(rows2))
    print(write_reports(cfg2.output_root, rows2))
